# Enforce Thermomix Tag Rule in Bucket + CSV

This notebook scans all recipe JSON files in the bucket and enforces:
- If a recipe has the `thermomix` tag, keep only that tag.
- Set tool tags to only `thermomix` for that recipe.
- Apply the same rule to `recipes_index.csv`.

Run in `dry_run=True` first, then re-run with `dry_run=False` to apply changes.


In [1]:
from __future__ import annotations

import csv
import importlib
import io
import json
import os
import re
import sys
from pathlib import Path


def _find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    markers = ('app.py', 'requirements.txt', 'env.prod', '.git')

    for root in [candidate, *candidate.parents]:
        if not (root / 'src').exists():
            continue
        if any((root / marker).exists() for marker in markers):
            return root

    for root in [candidate, *candidate.parents]:
        if (root / 'src').exists():
            return root

    raise RuntimeError('Could not find project root (expected at least a src/ folder).')


def _load_simple_env_file(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key:
            os.environ.setdefault(key, value)


def _normalize_path_env(var_name: str, project_root: Path) -> None:
    value = (os.getenv(var_name) or '').strip()
    if not value:
        return
    path = Path(value)
    if not path.is_absolute():
        path = (project_root / path).resolve()
    os.environ[var_name] = str(path)


PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

_load_simple_env_file(PROJECT_ROOT / 'env.prod')
_load_simple_env_file(PROJECT_ROOT / 'secrets' / 'env.prod')
for _var in ('API_BUCKET_KEY_FILE', 'GOOGLE_APPLICATION_CREDENTIALS'):
    _normalize_path_env(_var, PROJECT_ROOT)

import src.gcs_storage as _gcs_storage
_gcs_storage = importlib.reload(_gcs_storage)

download_bytes = _gcs_storage.download_bytes
upload_bytes = _gcs_storage.upload_bytes
get_bucket = _gcs_storage.get_bucket
storage_client = _gcs_storage.storage_client
bucket_name = _gcs_storage.bucket_name

RECIPES_PREFIX = (os.getenv('RECETAS_RECIPES_PREFIX') or 'recipes').strip('/ ')
RECIPE_INDEX_BLOB = (os.getenv('RECETAS_RECIPE_INDEX_BLOB') or f'{RECIPES_PREFIX}/recipes_index.csv').strip('/ ')
THERMOMIX_TAG = 'thermomix'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('Bucket =', bucket_name())
print('Recipes prefix =', RECIPES_PREFIX)
print('Index CSV blob =', RECIPE_INDEX_BLOB)
print('API_BUCKET_KEY_FILE =', os.getenv('API_BUCKET_KEY_FILE', ''))


PROJECT_ROOT = /home/rafael/E/Python_Projects/Recetas_webapp
Bucket = recetas-bucket-prod
Recipes prefix = recipes
Index CSV blob = recipes/recipes_index.csv
API_BUCKET_KEY_FILE = /home/rafael/E/Python_Projects/Recetas_webapp/secrets/recetas-webapp-prod-sa.json


In [2]:
def _slugify(value: str) -> str:
    normalized = re.sub(r'[^a-z0-9]+', '-', str(value or '').strip().lower())
    return normalized.strip('-') or 'receta'


def _parse_tag_list(value: object) -> list[str]:
    if isinstance(value, (list, tuple, set)):
        raw_values = [str(item or '').strip() for item in value]
    elif isinstance(value, str):
        raw_values = [chunk.strip() for chunk in value.split(',')]
    else:
        raw_values = []

    out: list[str] = []
    seen: set[str] = set()
    for raw in raw_values:
        lowered = raw.lower()
        if not lowered or lowered in seen:
            continue
        seen.add(lowered)
        out.append(lowered)
    return out


def _enforce_payload_rule(payload: dict[str, object]) -> tuple[bool, bool]:
    tags = _parse_tag_list(payload.get('Tags') if 'Tags' in payload else payload.get('tags'))
    tools = _parse_tag_list(payload.get('Tools') if 'Tools' in payload else payload.get('tools'))

    has_thermomix = THERMOMIX_TAG in tags
    if not has_thermomix:
        return False, False

    changed = tags != [THERMOMIX_TAG] or tools != [THERMOMIX_TAG]

    if 'Tags' in payload:
        payload['Tags'] = [THERMOMIX_TAG]
    if 'tags' in payload:
        payload['tags'] = [THERMOMIX_TAG]
    if 'Tags' not in payload and 'tags' not in payload:
        payload['Tags'] = [THERMOMIX_TAG]

    if 'Tools' in payload:
        payload['Tools'] = [THERMOMIX_TAG]
    if 'tools' in payload:
        payload['tools'] = [THERMOMIX_TAG]
    if 'Tools' not in payload and 'tools' not in payload:
        payload['Tools'] = [THERMOMIX_TAG]

    return changed, True


def _update_index_csv(*, dry_run: bool, thermomix_slugs: set[str]) -> dict[str, object]:
    try:
        raw_csv = download_bytes(RECIPE_INDEX_BLOB)
    except FileNotFoundError:
        return {
            'csv_found': False,
            'rows_total': 0,
            'rows_changed': 0,
            'missing_thermomix_rows': sorted(thermomix_slugs),
        }

    csv_text = raw_csv.decode('utf-8-sig')
    reader = csv.DictReader(io.StringIO(csv_text))
    fieldnames = list(reader.fieldnames or [])
    if 'tags' not in fieldnames:
        fieldnames.append('tags')
    if 'tags_tools' not in fieldnames:
        fieldnames.append('tags_tools')

    rows: list[dict[str, str]] = []
    rows_changed = 0
    seen_csv_slugs: set[str] = set()

    for raw_row in reader:
        row = {str(k or '').strip(): str(v or '').strip() for k, v in (raw_row or {}).items() if str(k or '').strip()}
        slug = _slugify(row.get('slug') or row.get('recipe_name') or '')
        if slug:
            seen_csv_slugs.add(slug)

        row_tags = _parse_tag_list(row.get('tags'))
        should_enforce = (slug in thermomix_slugs) or (THERMOMIX_TAG in row_tags)
        if should_enforce:
            prev_tags = row.get('tags', '').strip()
            prev_tools = row.get('tags_tools', '').strip()
            row['tags'] = THERMOMIX_TAG
            row['tags_tools'] = THERMOMIX_TAG
            if prev_tags != row['tags'] or prev_tools != row['tags_tools']:
                rows_changed += 1

        rows.append(row)

    if rows_changed and not dry_run:
        output = io.StringIO()
        writer = csv.DictWriter(output, fieldnames=fieldnames, extrasaction='ignore', lineterminator='\n')
        writer.writeheader()
        for row in rows:
            writer.writerow({field: str(row.get(field) or '') for field in fieldnames})
        upload_bytes(
            output.getvalue().encode('utf-8'),
            RECIPE_INDEX_BLOB,
            content_type='text/csv; charset=utf-8',
            cache_seconds=0,
        )

    missing_thermomix_rows = sorted(thermomix_slugs - seen_csv_slugs)
    return {
        'csv_found': True,
        'rows_total': len(rows),
        'rows_changed': rows_changed,
        'missing_thermomix_rows': missing_thermomix_rows,
    }


def enforce_thermomix_rule(*, dry_run: bool = True) -> dict[str, object]:
    client = storage_client()
    bucket = get_bucket()

    prefix = f'{RECIPES_PREFIX}/'
    scanned_json = 0
    changed_json = 0
    thermomix_slugs: set[str] = set()
    errors: list[str] = []

    for blob in client.list_blobs(bucket, prefix=prefix):
        blob_name = str(getattr(blob, 'name', '') or '').strip()
        if not blob_name.endswith('.json'):
            continue
        if blob_name == RECIPE_INDEX_BLOB:
            continue

        scanned_json += 1
        slug = _slugify(Path(blob_name).stem)

        try:
            raw = blob.download_as_bytes(client=client)
            payload = json.loads(raw.decode('utf-8'))
            if not isinstance(payload, dict):
                errors.append(f'{blob_name}: JSON root is not an object')
                continue
        except Exception as exc:  # noqa: BLE001
            errors.append(f'{blob_name}: {exc.__class__.__name__}: {exc}')
            continue

        changed, has_thermomix = _enforce_payload_rule(payload)
        if has_thermomix:
            thermomix_slugs.add(slug)

        if not changed:
            continue

        changed_json += 1
        if not dry_run:
            body = (json.dumps(payload, ensure_ascii=False, indent=2) + '\n').encode('utf-8')
            upload_bytes(
                body,
                blob_name,
                content_type='application/json; charset=utf-8',
                cache_seconds=0,
            )

    csv_report = _update_index_csv(dry_run=dry_run, thermomix_slugs=thermomix_slugs)

    report = {
        'dry_run': bool(dry_run),
        'bucket': bucket.name,
        'recipes_prefix': RECIPES_PREFIX,
        'index_blob': RECIPE_INDEX_BLOB,
        'scanned_json': scanned_json,
        'thermomix_recipes': len(thermomix_slugs),
        'json_changed': changed_json,
        'csv_found': csv_report['csv_found'],
        'csv_rows_total': csv_report['rows_total'],
        'csv_rows_changed': csv_report['rows_changed'],
        'missing_thermomix_rows': csv_report['missing_thermomix_rows'],
        'errors_count': len(errors),
        'errors_sample': errors[:20],
    }
    return report


In [3]:
# Dry run first
# This only reports what would change.
dry_run = True
report = enforce_thermomix_rule(dry_run=dry_run)
report


{'dry_run': True,
 'bucket': 'recetas-bucket-prod',
 'recipes_prefix': 'recipes',
 'index_blob': 'recipes/recipes_index.csv',
 'scanned_json': 7779,
 'thermomix_recipes': 7169,
 'json_changed': 7169,
 'csv_found': True,
 'csv_rows_total': 7779,
 'csv_rows_changed': 7169,
 'missing_thermomix_rows': [],
 'errors_count': 0,
 'errors_sample': []}

In [4]:
# Apply changes
# Set to False only after reviewing the dry run report above.
dry_run = False
report = enforce_thermomix_rule(dry_run=dry_run)
report


{'dry_run': False,
 'bucket': 'recetas-bucket-prod',
 'recipes_prefix': 'recipes',
 'index_blob': 'recipes/recipes_index.csv',
 'scanned_json': 7779,
 'thermomix_recipes': 7169,
 'json_changed': 7169,
 'csv_found': True,
 'csv_rows_total': 7779,
 'csv_rows_changed': 7169,
 'missing_thermomix_rows': [],
 'errors_count': 0,
 'errors_sample': []}